In [1]:
import glob
import os
from pathlib import Path

from QuantNado import BamStore

# Make Zarr Dataset
This notebook makes, loads, and explores the Zarr xarray dataset created from BAM files.

In [ ]:
# Setup paths and parameters
data_dir = "/Users/catherine/work/project/QuantNado/data/2025-12-17_menin_inh_24hr"
metadata_files = sorted(glob.glob(f"{data_dir}/metadata_*.csv"))
chrom_sizes_path = "/Users/catherine/work/project/QuantNado/data/hg38/hg38.chrom.sizes"
bam_files = sorted(glob.glob(f"{data_dir}/seqnado_output/**/aligned/SEM*.bam"))
print("Found", len(bam_files), "BAM Files")

Found 20 BAM Files


## Process BAM files to Zarr

Process each BAM file using the parallel chromosome processing:

In [3]:
BamStore.from_bam_files(
    bam_files=bam_files,
    chromsizes=chrom_sizes_path,
    store_path="dataset_full",
    metadata=metadata_files,
    filter_chromosomes=True,
    max_workers=8,
    overwrite=True,
    sample_column="sample_id",
    backend="zarr",
    chunk_len=64_000,
    log_file="bam_store.log",
)

2025-12-19 23:44:50 [INFO] Combining 3 metadata files
2025-12-19 23:44:50 [INFO] Reading metadata file: metadata_atac.csv
2025-12-19 23:44:50 [INFO] Reading metadata file: metadata_chip.csv
2025-12-19 23:44:50 [INFO] Reading metadata file: metadata_rna.csv
2025-12-19 23:44:50 [INFO] Processing 20 BAM files into unified dataset: 'dataset_full'
2025-12-19 23:44:50 [INFO] Loaded 25 chromosomes from /Users/catherine/work/project/QuantNado/data/hg38/hg38.chrom.sizes
2025-12-19 23:44:50 [INFO] Ragged layout: 25 contigs, total positions 3,088,286,401, chunk_len=64000
2025-12-19 23:44:50 [WARNING] Deleting existing store at: dataset_full.zarr
2025-12-19 23:46:03 [INFO] Found 3 assays: ['ATAC', 'ChIP', 'RNA']
2025-12-19 23:46:03 [INFO] Total samples across all assays: 20
2025-12-19 23:46:03 [INFO]   ATAC: 2 samples
2025-12-19 23:46:03 [INFO]   ChIP: 12 samples
2025-12-19 23:46:03 [INFO]   RNA: 6 samples
2025-12-19 23:46:03 [INFO] Processing [1/20] ATAC sample SEM-DMSO from 'atac/aligned/SEM-DMS

# Load Dataset

In [4]:
ds = BamStore.open("dataset_full", backend="zarr")
ds

2025-12-20 01:20:35 [INFO] Opening zarr store at: dataset_full.zarr


<xarray.Dataset> Size: 148GB
Dimensions:        (sample: 20, position_flat: 3088286401, contig: 25)
Coordinates:
  * sample         (sample) int64 160B 0 1 2 3 4 5 6 7 ... 13 14 15 16 17 18 19
  * position_flat  (position_flat) int64 25GB 0 1 2 ... 3088286399 3088286400
  * contig         (contig) int64 200B 0 1 2 3 4 5 6 7 ... 18 19 20 21 22 23 24
    contig_length  (contig) int64 200B dask.array<chunksize=(25,), meta=np.ndarray>
    contig_offset  (contig) int64 200B dask.array<chunksize=(25,), meta=np.ndarray>
Data variables:
    signal         (sample, position_flat) uint16 124GB dask.array<chunksize=(1, 64000), meta=np.ndarray>
Attributes: (12/17)
    assay_by_sample:         ['ATAC', 'ATAC', 'ChIP', 'ChIP', 'ChIP', 'ChIP',...
    metadata_control:        ['', '', 'Input', 'Input', 'Input', 'Input', 'In...
    metadata_ip:             ['', '', 'H3K27Ac', 'H3K27Ac', 'MLL', 'MLL', 'Me...
    metadata_replicate:      ['', '', '', '', '', '', '', '', '', '', '', '',...
    metadata_scaling_group:  ['default', 'default', 'default', 'default', 'de...
    metadata_timepoint:      ['24hr', '24hr', '24hr', '24hr', '24hr', '24hr',...
    ...                      ...
    assays:                  ATAC,ChIP,RNA
    sample_names:            ['SEM-DMSO', 'SEM-MENi', 'SEM-DMSO-H3K27Ac_H3K27...
    contig_names:            ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6',...
    structure:               ragged (sample × position_flat with contig offsets)
    bin_size:                1
    average_sparsity:        71.70%